# 1. Imports

In [1]:
import pandas as pd
from pathlib import Path
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

pd.set_option('display.max_columns', None)

# 2. Funções

## 2.1. Função para pegar os eventos de uma temporada nos arquivos parquet

In [2]:
def get_season_events_parquet_file_paths(events_competition_season_folder_path):
    
    season_events_parquet_file_paths = [
        str(Path(events_competition_season_folder_path) / season_event_parquet_file) 
        for season_event_parquet_file in os.listdir(events_competition_season_folder_path) 
        if season_event_parquet_file.endswith('.parquet')
        ]
    
    return season_events_parquet_file_paths

# 3. Preparação dos dados

## 3.1. Criação da Sessão Spark

In [3]:
# Criação da sessão Spark local
spark = SparkSession.builder.master("local[*]").appName("season_database").getOrCreate()

## 3.2. Criação do df para pegar os eventos de todas as partidas da temporada de 2022-2023 da Premier League

(dps pode ser interessante levar a parte do schema dos jogadores e da bola p etapa de extração)

In [4]:
events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / "1" / "2022-2023")

season_events_parquet_file_paths = get_season_events_parquet_file_paths(events_competition_season_folder_path)

# Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
df_events = spark.read.parquet(*season_events_parquet_file_paths)

df_events = df_events.withColumnsRenamed({
    "id": "eventId",
    "player.id": "eventPlayer.id",
    "player.name": "eventPlayer.name",
    "team.id": "eventTeam.id",
    "team.name": "eventTeam.name",
})

In [5]:
df_events.select('homePlayers', 'awayPlayers', 'balls').first()

Row(homePlayers='[{"speed": 0.452, "y": 25.001, "x": 15.746, "player": {"id": 292, "name": "Adam Smith"}, "visibility": "VISIBLE", "confidence": "LOW", "jerseyNum": null}, {"speed": 0.271, "y": -14.281, "x": 1.674, "player": {"id": 312, "name": "Dominic Solanke"}, "visibility": "VISIBLE", "confidence": "MEDIUM", "jerseyNum": null}, {"speed": 0.865, "y": -19.598, "x": 14.398, "player": {"id": 6994, "name": "Jordan Zemura"}, "visibility": "ESTIMATED", "confidence": "LOW", "jerseyNum": null}, {"speed": 0.163, "y": 5.119, "x": 8.692, "player": {"id": 7230, "name": "Ben Pearson"}, "visibility": "VISIBLE", "confidence": "MEDIUM", "jerseyNum": null}, {"speed": 0.792, "y": -3.978, "x": 18.401, "player": {"id": 289, "name": "Lloyd Kelly"}, "visibility": "ESTIMATED", "confidence": "LOW", "jerseyNum": null}, {"speed": 0.833, "y": 12.426, "x": 15.262, "player": {"id": 295, "name": "Jefferson Lerma"}, "visibility": "ESTIMATED", "confidence": "LOW", "jerseyNum": null}, {"speed": 0.775, "y": -8.477, 

In [6]:
events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / "1" / "2022-2023")

season_events_parquet_file_paths = get_season_events_parquet_file_paths(events_competition_season_folder_path)

# Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
df_events = spark.read.parquet(*season_events_parquet_file_paths)

df_events = df_events.withColumnsRenamed({
    "id": "eventId",
    "player.id": "eventPlayer.id",
    "player.name": "eventPlayer.name",
    "team.id": "eventTeam.id",
    "team.name": "eventTeam.name",
})

# Schema em Pyspark para poder parsear o json dos dados de tracking dos jogadores que está como string
players_schema = ArrayType(
    StructType([
        #StructField("speed", FloatType(), True),
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("player", StructType([
            StructField("id", IntegerType(), True), 
            StructField("name", StringType(), True)]), 
            True),
        StructField("visibility", StringType(), True),
        StructField("confidence", StringType(), True),
        #StructField("jerseyNum", StringType(), True)      
    ])
)

# Schema em Pyspark para poder parsear o json dos dados de tracking da bola que está como string
balls_schema = ArrayType(
    StructType([
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("z", FloatType(), True),
        StructField("visibility", StringType(), True)
    ])
)

details_schema = MapType(StringType(), StringType())

df_events = df_events.withColumns({
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time mandante
    "homePlayers_parsed": F.from_json("homePlayers", players_schema),
    
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time adversário
    "awayPlayers_parsed": F.from_json("awayPlayers", players_schema),

    # Cria coluna com json parseado para dicionário para dados de tracking da bola
    "balls_parsed": F.from_json("balls", balls_schema),

    "details_parsed": F.from_json("details", details_schema)

}).drop('homePlayers', 'awayPlayers', 'balls', 'details')

df_events = df_events.select(
    'competitionId',
    'season', # dps mudar pra seasonId se necessário
    'gameId',
    'eventId',
    'eventType',
    'eventTypeDescription',
    'period',
    #'periodDescription',
    'startFormattedGameClock',
    'startGameClock',
    #'details_parsed',
    'homeTeam',
    F.col('`eventPlayer.id`').alias('eventPlayerId'),
    F.col('`eventPlayer.name`').alias('eventPlayerName'),
    F.col('`eventTeam.id`').alias('eventTeamId'), 
    F.col('`eventTeam.name`').alias('eventTeamName'), 
    'homePlayers_parsed', 
    'awayPlayers_parsed', 
    'balls_parsed'
)

In [7]:
df_events.filter(F.size("homePlayers_parsed") == 0).show()

+-------------+---------+------+--------------------+---------+--------------------+------+-----------------------+--------------+--------+-------------+--------------------+-----------+--------------------+------------------+------------------+------------+
|competitionId|   season|gameId|             eventId|eventType|eventTypeDescription|period|startFormattedGameClock|startGameClock|homeTeam|eventPlayerId|     eventPlayerName|eventTeamId|       eventTeamName|homePlayers_parsed|awayPlayers_parsed|balls_parsed|
+-------------+---------+------+--------------------+---------+--------------------+------+-----------------------+--------------+--------+-------------+--------------------+-----------+--------------------+------------------+------------------+------------+
|            1|2022-2023|  4455|cf04dbb06a6c37975...|       CH|           Challenge|     1|                  16:36|           996|   false|          393|       Harrison Reed|         54|              Fulham|                

In [8]:
df_events.filter(F.size("awayPlayers_parsed") == 0).show()

+-------------+---------+------+--------------------+---------+--------------------+------+-----------------------+--------------+--------+-------------+--------------------+-----------+--------------------+------------------+------------------+------------+
|competitionId|   season|gameId|             eventId|eventType|eventTypeDescription|period|startFormattedGameClock|startGameClock|homeTeam|eventPlayerId|     eventPlayerName|eventTeamId|       eventTeamName|homePlayers_parsed|awayPlayers_parsed|balls_parsed|
+-------------+---------+------+--------------------+---------+--------------------+------+-----------------------+--------------+--------+-------------+--------------------+-----------+--------------------+------------------+------------------+------------+
|            1|2022-2023|  4455|cf04dbb06a6c37975...|       CH|           Challenge|     1|                  16:36|           996|   false|          393|       Harrison Reed|         54|              Fulham|                

In [9]:
df_events.filter(F.size("balls_parsed") == 0).show()

+-------------+---------+------+--------------------+---------+--------------------+------+-----------------------+--------------+--------+-------------+----------------+-----------+---------------+--------------------+--------------------+------------+
|competitionId|   season|gameId|             eventId|eventType|eventTypeDescription|period|startFormattedGameClock|startGameClock|homeTeam|eventPlayerId| eventPlayerName|eventTeamId|  eventTeamName|  homePlayers_parsed|  awayPlayers_parsed|balls_parsed|
+-------------+---------+------+--------------------+---------+--------------------+------+-----------------------+--------------+--------+-------------+----------------+-----------+---------------+--------------------+--------------------+------------+
|            1|2022-2023|  4438|55cc16113c5f9fc11...|       PA|                Pass|     1|                  02:51|           171|   false|          402|      Danny Ings|          3|    Aston Villa|[{14.151, 26.691,...|[{-17.519, 11.832..

In [10]:
df_events.select('homePlayers_parsed', 'awayPlayers_parsed', 'balls_parsed').first()

Row(homePlayers_parsed=[Row(x=15.746000289916992, y=25.000999450683594, player=Row(id=292, name='Adam Smith'), visibility='VISIBLE', confidence='LOW'), Row(x=1.6740000247955322, y=-14.281000137329102, player=Row(id=312, name='Dominic Solanke'), visibility='VISIBLE', confidence='MEDIUM'), Row(x=14.39799976348877, y=-19.597999572753906, player=Row(id=6994, name='Jordan Zemura'), visibility='ESTIMATED', confidence='LOW'), Row(x=8.692000389099121, y=5.11899995803833, player=Row(id=7230, name='Ben Pearson'), visibility='VISIBLE', confidence='MEDIUM'), Row(x=18.400999069213867, y=-3.9779999256134033, player=Row(id=289, name='Lloyd Kelly'), visibility='ESTIMATED', confidence='LOW'), Row(x=15.26200008392334, y=12.425999641418457, player=Row(id=295, name='Jefferson Lerma'), visibility='ESTIMATED', confidence='LOW'), Row(x=-0.15299999713897705, y=-8.47700023651123, player=Row(id=7966, name='Kieffer Moore'), visibility='VISIBLE', confidence='LOW'), Row(x=39.97700119018555, y=-0.12099999934434891,

In [11]:
print('Quantidade de linhas:', df_events.count())

Quantidade de linhas: 945154


## 3.2. Obter jogos da temporada e ajustar identificação do mandante/adversário

(dps pode ser interessante levar isso p etapa de extração)

In [12]:
games_path = str(Path().resolve().parent.parent / "data" / "games.csv")

df_games = spark.read.csv(games_path, header=True)

df_games_raw = df_games.withColumnRenamed("id","gameId").filter(F.col('season') == '2022-2023')

# se venueType == TEAM_HOME, (homeTeam.id == team.id e homeTeam.name == team.name) e (opponentTeam.id == opponentTeam.id e opponentTeam.name == opponentTeam.name)
# se venueType == OPPONENT_HOME, (homeTeam.id == opponentTeam.id e homeTeam.name == opponentTeam.name) e (opponentTeam.id == team.id e opponentTeam.name == team.name)
df_games = (
    df_games_raw.withColumns({
    "homeTeamId": F.when(F.col('venueType') == 'TEAM_HOME', F.col('`team.id`')).otherwise(F.col('`opponentTeam.id`')),
    "homeTeamName": F.when(F.col('venueType') == 'TEAM_HOME', F.col('`team.name`')).otherwise(F.col('`opponentTeam.name`')),

    "opponentTeamId": F.when(F.col('venueType') == 'TEAM_HOME', F.col('`opponentTeam.id`')).otherwise(F.col('`opponentTeam.id`')),
    "opponentTeamName": F.when(F.col('venueType') == 'TEAM_HOME', F.col('`opponentTeam.name`')).otherwise(F.col('`opponentTeam.name`')),
    }).select(
        'gameId', 
        'date',
        'season',
        F.col('`competition.id`').alias('competitionId'),
        F.col('`competition.name`').alias('competitionName'),
        'homeTeamId',
        'homeTeamName',
        'opponentTeamId',
        'opponentTeamName',
        #F.col('teamExtraTimeStartSide').alias('homeTeamExtraTimeStartSide'), 
        F.col('teamStartSide').alias('homeTeamStartSide'),
        F.col('`stadium.name`').alias('stadiumName'), 
        F.col('`stadium.length`').cast("float").alias('stadiumLength'), 
        F.col('`stadium.width`').cast("float").alias('stadiumWidth')
    )
)

df_games.show(5)

+------+----------+---------+-------------+---------------+----------+--------------------+--------------+--------------------+-----------------+-------------+-------------+------------+
|gameId|      date|   season|competitionId|competitionName|homeTeamId|        homeTeamName|opponentTeamId|    opponentTeamName|homeTeamStartSide|  stadiumName|stadiumLength|stadiumWidth|
+------+----------+---------+-------------+---------------+----------+--------------------+--------------+--------------------+-----------------+-------------+-------------+------------+
|  4447|2022-08-13|2022-2023|            1| Premier League|         3|         Aston Villa|             8|             Everton|             Left|   Villa Park|        105.0|        68.0|
|  4760|2023-04-25|2022-2023|            1| Premier League|        20|Wolverhampton Wan...|            20|Wolverhampton Wan...|             Left|     Molineux|        105.0|        68.0|
|  4451|2022-08-15|2022-2023|            1| Premier League|      

## 4. Junção dos dados dos Jogos + Eventos em uma tabela

In [13]:
df_games_events = df_events.join(df_games.drop('season', 'competitionId', 'competitionName'), on = "gameId", how='left')

df_games_events = (
    df_games_events
    .withColumn(
        'homeTeamAttackDirection',
            F.when(
                ((F.col('period') == 1) & (F.col('homeTeamStartSide') == 'Right')) | 
                ((F.col('period') == 2) & (F.col('homeTeamStartSide') == 'Left')), 
                'Left'
            )
            .when(
                ((F.col('period') == 1) & (F.col('homeTeamStartSide') == 'Left')) | 
                ((F.col('period') == 2) & (F.col('homeTeamStartSide') == 'Right')), 
                'Right'
            )
        )
    .withColumn(
            'awayTeamAttackDirection',
            F.when(F.col('homeTeamAttackDirection') == 'Right', 'Left')
            .when(F.col('homeTeamAttackDirection') == 'Left', 'Right')
        )
    .drop('homeTeamStartSide')
)

df_games_events.show(5)

+------+-------------+---------+--------------------+------------+--------------------+------+-----------------------+--------------+--------+-------------+---------------+-----------+-------------+--------------------+--------------------+--------------------+----------+----------+---------------+--------------+----------------+----------------+-------------+------------+-----------------------+-----------------------+
|gameId|competitionId|   season|             eventId|   eventType|eventTypeDescription|period|startFormattedGameClock|startGameClock|homeTeam|eventPlayerId|eventPlayerName|eventTeamId|eventTeamName|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|      date|homeTeamId|   homeTeamName|opponentTeamId|opponentTeamName|     stadiumName|stadiumLength|stadiumWidth|homeTeamAttackDirection|awayTeamAttackDirection|
+------+-------------+---------+--------------------+------------+--------------------+------+-----------------------+--------------+--------+----------

In [14]:
df_games_events.groupby('period').count().show()

+------+------+
|period| count|
+------+------+
|     1|479855|
|     2|465299|
+------+------+



In [15]:
# variável que indica o time com a posse
df_games_events.groupBy('homeTeam').count().show()

+--------+------+
|homeTeam| count|
+--------+------+
|    NULL|  6918|
|    true|472671|
|   false|465565|
+--------+------+



In [16]:
df_events.filter(
    (F.size("balls_parsed") != 0) |
    (F.size("homePlayers_parsed") != 0) |
    (F.size("awayPlayers_parsed") != 0)
    ).show()

+-------------+---------+------+--------------------+------------+--------------------+------+-----------------------+--------------+--------+-------------+----------------+-----------+---------------+--------------------+--------------------+--------------------+
|competitionId|   season|gameId|             eventId|   eventType|eventTypeDescription|period|startFormattedGameClock|startGameClock|homeTeam|eventPlayerId| eventPlayerName|eventTeamId|  eventTeamName|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|
+-------------+---------+------+--------------------+------------+--------------------+------+-----------------------+--------------+--------+-------------+----------------+-----------+---------------+--------------------+--------------------+--------------------+
|            1|2022-2023|  4438|9e6a498f54bad910e...|          PA|                Pass|     1|                  00:00|             0|   false|          402|      Danny Ings|          3|    Aston Villa|[{15

In [17]:
df_games_events = (
    df_games_events
    .filter(
        # filtro para garantir apenas eventos das partidas no 1º e 2º tempo
        (F.col('period').isin([1,2])) &
        # filtro para não considerar tracking da bola e jogadores home/away que n tem tracking (dps podemos pensar em imputar)
        (F.size("balls_parsed") != 0) & (F.size("homePlayers_parsed") != 0) & (F.size("awayPlayers_parsed") != 0)
    ) 
    .dropna(subset='homeTeam') # drop nos eventos onde nenhum dos dois times tem a posse    
    )

In [18]:
df_games_events.filter(
    (F.size("balls_parsed") == 0) |
    (F.size("homePlayers_parsed") == 0) |
    (F.size("awayPlayers_parsed") == 0)
    ).show()

+------+-------------+------+-------+---------+--------------------+------+-----------------------+--------------+--------+-------------+---------------+-----------+-------------+------------------+------------------+------------+----+----------+------------+--------------+----------------+-----------+-------------+------------+-----------------------+-----------------------+
|gameId|competitionId|season|eventId|eventType|eventTypeDescription|period|startFormattedGameClock|startGameClock|homeTeam|eventPlayerId|eventPlayerName|eventTeamId|eventTeamName|homePlayers_parsed|awayPlayers_parsed|balls_parsed|date|homeTeamId|homeTeamName|opponentTeamId|opponentTeamName|stadiumName|stadiumLength|stadiumWidth|homeTeamAttackDirection|awayTeamAttackDirection|
+------+-------------+------+-------+---------+--------------------+------+-----------------------+--------------+--------+-------------+---------------+-----------+-------------+------------------+------------------+------------+----+-------

## Normalização do ataque sempre pra direita

In [19]:
# time com a posse está atacando e time sem está defendendo

df_games_events_tracking = (
    df_games_events
    # time atacando = se o time da casa tiver a posse, pega tracking home, se não pega tracking away
    .withColumn(
        "attackingPlayers",
        F.when(F.col("homeTeam"), F.col("homePlayers_parsed"))
        .otherwise(F.col("awayPlayers_parsed"))
    )
    # time defendendo = se o time da casa tiver a posse, pega tracking away, se não pega tracking home
    .withColumn(
        "defendingPlayers",
        F.when(F.col("homeTeam"), F.col("awayPlayers_parsed"))
        .otherwise(F.col("homePlayers_parsed"))
    )
    .withColumn(
        "attackingDirection",
        F.when(F.col("homeTeam"), F.col("homeTeamAttackDirection"))
        .otherwise(F.col("awayTeamAttackDirection"))
    )
    # flag para normalização (para tratar ataque sempre pra direita)
    .withColumn(
        "need_side_revert",
        F.col("attackingDirection") == "Left"
    )
    .drop(
        'homePlayers_parsed',  
        'awayPlayers_parsed',
        'homeTeamAttackDirection',
        'awayTeamAttackDirection'
    )
)

df_games_events_tracking.show()

+------+-------------+---------+--------------------+------------+--------------------+------+-----------------------+--------------+--------+-------------+----------------+-----------+---------------+--------------------+----------+----------+---------------+--------------+----------------+----------------+-------------+------------+--------------------+--------------------+------------------+----------------+
|gameId|competitionId|   season|             eventId|   eventType|eventTypeDescription|period|startFormattedGameClock|startGameClock|homeTeam|eventPlayerId| eventPlayerName|eventTeamId|  eventTeamName|        balls_parsed|      date|homeTeamId|   homeTeamName|opponentTeamId|opponentTeamName|     stadiumName|stadiumLength|stadiumWidth|    attackingPlayers|    defendingPlayers|attackingDirection|need_side_revert|
+------+-------------+---------+--------------------+------------+--------------------+------+-----------------------+--------------+--------+-------------+--------------

In [20]:
players_tracking_norm = (
        lambda p: F.struct(
            # reversão do eixo horizontal qnd necessário
            F.when(F.col("need_side_revert"), -p["x"])
            .otherwise(p["x"])
            .alias("x"),
            # variáveis restantes mantém igual
            p["y"].alias("y"),            
            p["player"].alias("player"),
            p["visibility"].alias("visibility"),
            p["confidence"].alias("confidence")
        )
)

balls_norm = (
    F.transform(
        "balls_parsed",
        lambda b: F.struct(
            # reversão do eixo horizontal qnd necessário
            F.when(F.col("need_side_revert"), -b["x"])
            .otherwise(b["x"])
            .alias("x"),
            # variáveis restantes mantém igual            
            b["y"].alias("y"),
            b["z"].alias("z"),            
            b["visibility"].alias("visibility")
        )
    )
)

# df para normalizar os atacantes, defensores e bola sempre atacando do lado direito
df_games_events_tracking_norm = (
    df_games_events_tracking
    # normalizar atacantes
    .withColumns({
        "attackingPlayersNorm": F.transform("attackingPlayers", players_tracking_norm),
        # normalizar defensores
        "defendingPlayersNorm": F.transform("defendingPlayers", players_tracking_norm),
        # normalizar a bola
        "ballsNorm": balls_norm
    }).drop(
        'attackingPlayers', 
        'defendingPlayers',
        'balls_parsed',
        'attackingDirection',
        'need_side_revert'
        )
)

#### Métrica 1: Distância percorrida no campo (medida pela menor distância entre os escanteios)

In [21]:
def euclidean_dist(x1, y1, x2, y2):
    return F.sqrt(
        F.pow(x1 - x2, 2) +
        F.pow(y1 - y2, 2)
    )

In [22]:
# Extremos esquerdo e direito do campo no plano cartesiano
left_x = -F.col('stadiumLength') / 2
right_x = F.col('stadiumLength') / 2

# Extremos superior e inferior do campo no plano cartesiano
top_y = F.col('stadiumWidth') / 2
bottom_y = -F.col('stadiumWidth') / 2

# Eixos x e y da posição da bola a partir dos dados de tracking
ball_x = F.get("ballsNorm", 0)["x"]
ball_y = F.get("ballsNorm", 0)["y"]

# Menor distância euclidiana entre os escanteios do lado esquerdo até a bola
progression_distance = F.round(
    F.least(
        euclidean_dist(ball_x, ball_y, left_x, top_y),
        euclidean_dist(ball_x, ball_y, left_x, bottom_y)
    ), 2)

#### Métrica 2: Quantidade total de jogadores dos dois times (entre a bola e o gol)
#### Métrica 3: Vantagem numérica do ataque em relação à defesa (entre a bola e o gol)

In [23]:
df_games_events_players_ball_goal = (
    df_games_events_tracking_norm
     .withColumns({
        # Quantidade de jogadores do time mandante entre o gol esquerdo e a bola
        'attackers_between_ball_goal': (
            F.size(
                F.filter(
                    F.col('attackingPlayersNorm'),
                    lambda p: (
                        # jogadores no eixo x entre a bola e o gol direito
                        (p['x'] <= right_x) &
                        (p['x'] >= ball_x)
                    )
                )
            )
        ),
        'defenders_between_ball_goal': (
            F.size(
                F.filter(
                    F.col('defendingPlayersNorm'),
                    lambda p: (
                        # jogadores no eixo x entre a bola e o gol direito
                        (p['x'] <= right_x) &
                        (p['x'] >= ball_x)
                    )
                )
            )
        )
    })
)

df_games_events_players_ball_goal = (
    df_games_events_players_ball_goal.withColumns({
        'progression_distance': progression_distance,
        'total_players_between_ball_goal': F.col('attackers_between_ball_goal') + F.col('defenders_between_ball_goal'),
        'atk_def_advantage_between_ball_goal': F.col('attackers_between_ball_goal') - F.col('defenders_between_ball_goal')
    })
)

df_games_events_players_ball_goal.show()

+------+-------------+---------+--------------------+------------+--------------------+------+-----------------------+--------------+--------+-------------+----------------+-----------+---------------+----------+----------+---------------+--------------+----------------+----------------+-------------+------------+--------------------+--------------------+--------------------+---------------------------+---------------------------+--------------------+-------------------------------+-----------------------------------+
|gameId|competitionId|   season|             eventId|   eventType|eventTypeDescription|period|startFormattedGameClock|startGameClock|homeTeam|eventPlayerId| eventPlayerName|eventTeamId|  eventTeamName|      date|homeTeamId|   homeTeamName|opponentTeamId|opponentTeamName|     stadiumName|stadiumLength|stadiumWidth|attackingPlayersNorm|defendingPlayersNorm|           ballsNorm|attackers_between_ball_goal|defenders_between_ball_goal|progression_distance|total_players_between

### Criação da Ameaça pela Média das 3 componentes normalizadas com Min-Max

In [24]:
# Obtém mínimos e máximos das componentes
stats = (
    df_games_events_players_ball_goal
    .agg(
        F.min("progression_distance").alias("min_pd"),
        F.max("progression_distance").alias("max_pd"),

        F.min("total_players_between_ball_goal").alias("min_tp"),
        F.max("total_players_between_ball_goal").alias("max_tp"),

        F.min("atk_def_advantage_between_ball_goal").alias("min_adv"),
        F.max("atk_def_advantage_between_ball_goal").alias("max_adv")
    )
    .first()
)

df_games_events_players_ball_goal_norm = (
    df_games_events_players_ball_goal
    # Componentes normalizadas [0,1]
    .withColumns({
        # Progressão em campo em direção ao gol defendido
        "progression_distance_norm":
        F.round((F.col("progression_distance") - F.lit(stats["min_pd"])) / F.lit(stats["max_pd"] - stats["min_pd"]), 3),

        # Quantidade absoluta de jogadores entre a bola e o gol (1 - minmax por ser inversamente proporcional)
        "total_players_between_ball_goal_norm": F.round(1 - 
        (F.col("total_players_between_ball_goal") - F.lit(stats["min_tp"])) / F.lit(stats["max_tp"] - stats["min_tp"]), 3), 
        
        # Vantagem numérica do ataque em relação à defesa
        "atk_def_advantage_between_ball_goal_norm":
        F.round((F.col("atk_def_advantage_between_ball_goal") - F.lit(stats["min_adv"])) / F.lit(stats["max_adv"] - stats["min_adv"]), 3),

        # Threat score = média das 3 componentes
        "threat_score": F.round((
            F.col("progression_distance_norm") + F.col("total_players_between_ball_goal_norm") + F.col("atk_def_advantage_between_ball_goal_norm")
        ) / F.lit(3.0), 3)

    })
)

In [25]:
w = (
    Window
    .partitionBy(
        "gameId",
        "competitionId",
        "season",
        "homeTeam"
    )
    .orderBy("startGameClock")
)

df_threat_final = (
    df_games_events_players_ball_goal_norm
    .withColumn(
        "threat_score_delta",
        # F.round(
        #     F.coalesce(
        #         F.col("threat_score") - F.lag("threat_score").over(w),
        #         F.lit(0.0)
        #     ), 3
        # )
        F.round((F.col("threat_score") - F.lag("threat_score").over(w)), 3)
    )
)

In [27]:
df_threat_final.filter(
    (F.col('gameId') == 4438) &
    (F.col('competitionId') == 1) &
    (F.col('season') == '2022-2023') &
    (F.col('homeTeamName') == 'AFC Bournemouth')
    ).sort('startGameClock').show(50)

+------+-------------+---------+--------------------+------------+--------------------+------+-----------------------+--------------+--------+-------------+-----------------+-----------+---------------+----------+----------+---------------+--------------+----------------+----------------+-------------+------------+--------------------+--------------------+--------------------+---------------------------+---------------------------+--------------------+-------------------------------+-----------------------------------+-------------------------+------------------------------------+----------------------------------------+------------+------------------+
|gameId|competitionId|   season|             eventId|   eventType|eventTypeDescription|period|startFormattedGameClock|startGameClock|homeTeam|eventPlayerId|  eventPlayerName|eventTeamId|  eventTeamName|      date|homeTeamId|   homeTeamName|opponentTeamId|opponentTeamName|     stadiumName|stadiumLength|stadiumWidth|attackingPlayersNorm|de